# Logistic Regression: Binary Classification Lab

This notebook uses the public **Wisconsin Diagnostic Breast Cancer** dataset, distributed with scikit-learn. We will predict whether a tumor is malignant or benign and connect the workflow to the main concepts of logistic regression.

You will:

- inspect a binary classification dataset;
- fit logistic regression and interpret predicted probabilities;
- visualize a decision boundary using two features;
- see how logistic loss penalizes confident mistakes;
- use cross-validation to select a hyperparameter;
- evaluate the chosen model once on an independent test set.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    log_loss,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

## 1. Load and inspect the data

In [ ]:
cancer = load_breast_cancer(as_frame=True)
X = cancer.data
y = cancer.target

print(cancer.DESCR.split('\n')[0])
print(f'Feature matrix shape: {X.shape}')
print('Target labels:', dict(enumerate(cancer.target_names)))
display(y.value_counts().rename(index=dict(enumerate(cancer.target_names))).to_frame('count'))
X.head()

The target is binary: `0` means malignant and `1` means benign. Logistic regression will estimate $P(y = 1 id x)$, the probability that an observation belongs to class 1.

## 2. Reserve an independent test set

The test set is held aside before model selection. We use stratification so both classes retain approximately the same proportion in each split.

In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print(f'Development set: {X_dev.shape[0]} observations')
print(f'Independent test set: {X_test.shape[0]} observations')

## 3. Fit logistic regression and inspect probabilities

Feature scaling is important because the measurements have different numerical ranges. The pipeline fits the scaler using only the training data, then applies logistic regression.

In [ ]:
model = Pipeline([
    ('scaler', StandardScaler()),
    ('logistic_regression', LogisticRegression(max_iter=5_000, random_state=RANDOM_STATE)),
])
model.fit(X_dev, y_dev)

sample_probabilities = model.predict_proba(X_test.iloc[:10])[:, 1]
sample_predictions = model.predict(X_test.iloc[:10])
probability_table = pd.DataFrame({
    'P(benign = 1)': sample_probabilities,
    'predicted class': sample_predictions,
    'actual class': y_test.iloc[:10].to_numpy(),
})
probability_table

The default decision threshold is 0.5. A probability at or above 0.5 is classified as benign (class 1); otherwise it is classified as malignant (class 0).

## 4. Visualize a decision boundary

A two-feature model lets us plot the decision boundary. The full model above uses all 30 features; this smaller model is only for visualization.

In [ ]:
plot_features = ['mean radius', 'mean texture']
X_plot = X_dev[plot_features]

boundary_model = Pipeline([
    ('scaler', StandardScaler()),
    ('logistic_regression', LogisticRegression(random_state=RANDOM_STATE)),
])
boundary_model.fit(X_plot, y_dev)

fig, ax = plt.subplots(figsize=(8, 6))
DecisionBoundaryDisplay.from_estimator(
    boundary_model, X_plot, response_method='predict_proba',
    plot_method='contourf', alpha=0.35, cmap='RdYlBu', ax=ax
)
scatter = ax.scatter(
    X_plot['mean radius'], X_plot['mean texture'], c=y_dev,
    cmap='RdYlBu', edgecolor='black', linewidth=0.3
)
ax.set_xlabel('Mean radius')
ax.set_ylabel('Mean texture')
ax.set_title('Logistic Regression Probability Regions')
ax.legend(*scatter.legend_elements(), title='Class', labels=cancer.target_names)
plt.show()

## 5. Logistic loss

Log loss is small when the predicted probability agrees confidently with the true label. It becomes large when the model is confidently wrong.

In [ ]:
examples = pd.DataFrame({
    'true label y': [1, 1, 0, 0],
    'predicted P(y = 1 | x)': [0.99, 0.05, 0.01, 0.95],
})
probabilities = examples['predicted P(y = 1 | x)'].to_numpy()
labels = examples['true label y'].to_numpy()
examples['logistic loss'] = -(
    labels * np.log(probabilities) + (1 - labels) * np.log(1 - probabilities)
)
examples

## 6. Use cross-validation to select a model

We compare several values of `C`, the inverse regularization strength. The test set remains untouched while this decision is made.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
candidate_c_values = [0.01, 0.1, 1, 10]
cv_results = []

for c_value in candidate_c_values:
    candidate = Pipeline([
        ('scaler', StandardScaler()),
        ('logistic_regression', LogisticRegression(C=c_value, max_iter=5_000, random_state=RANDOM_STATE)),
    ])
    scores = cross_val_score(candidate, X_dev, y_dev, cv=cv, scoring='neg_log_loss')
    cv_results.append({
        'C': c_value,
        'mean validation log loss': -scores.mean(),
        'standard deviation': scores.std(),
    })

cv_results = pd.DataFrame(cv_results).sort_values('mean validation log loss')
cv_results

## 7. Evaluate the final model once on the test set

Select the `C` value that achieved the lowest mean validation log loss, train the final model on all development data, and then evaluate it once on the independent test set.

In [ ]:
best_c = cv_results.iloc[0]['C']
final_model = Pipeline([
    ('scaler', StandardScaler()),
    ('logistic_regression', LogisticRegression(C=best_c, max_iter=5_000, random_state=RANDOM_STATE)),
])
final_model.fit(X_dev, y_dev)

test_predictions = final_model.predict(X_test)
test_probabilities = final_model.predict_proba(X_test)[:, 1]

print(f'Selected C: {best_c}')
print(f'Test accuracy: {accuracy_score(y_test, test_predictions):.3f}')
print(f'Test log loss: {log_loss(y_test, test_probabilities):.3f}')
print('\nClassification report:\n')
print(classification_report(y_test, test_predictions, target_names=cancer.target_names))

ConfusionMatrixDisplay.from_predictions(y_test, test_predictions, display_labels=cancer.target_names, cmap='Blues')
plt.title('Final Model: Independent Test Set')
plt.show()

## Reflection

1. Which examples in the logistic-loss table have the largest loss, and why?
2. How would changing the classification threshold affect malignant and benign predictions?
3. Why is the test score reported only after selecting `C` with cross-validation?